# 🤖 Multi-Agent Quantitative Analysis System
### AAFA · CrewAI · Groq LLaMA 3.3 · Yahoo Finance · Firecrawl

> **v2 — Rate-Limit Resilient Edition.** Token budgeting, exponential backoff, chunked tool output and streaming-safe task prompts prevent hitting Groq free-tier limits.

---
## 📋 Table of Contents
1. [Phase 0 – Environment Setup & API Keys](#phase0)
2. [Phase 1 – Install Dependencies](#phase1)
3. [Phase 2 – Write Project Source Files](#phase2)
4. [Phase 3 – Shared Modules (Config · Database · Storage)](#phase3)
5. [Phase 4 – Agent Tools (Financial · Scraper)](#phase4)
6. [Phase 5 – Agents, Tasks & Crew](#phase5)
7. [Phase 6 – API Layer (FastAPI)](#phase6)
8. [Phase 7 – Run the Analysis Pipeline](#phase7)
9. [Phase 8 – Save All Outputs & Metadata](#phase8)


---
## ⚙️ Phase 0 – API Keys Setup <a id='phase0'></a>

### Required API Keys
| Key | Where to Get | Used For |
|-----|-------------|----------|
| `GROQ_API_KEY` | https://console.groq.com → Sign up → API Keys | LLM backbone (LLaMA 3.3 70B Versatile) |
| `FIRECRAWL_API_KEY` | https://www.firecrawl.dev → Sign up → Dashboard | Web scraping & news sentiment |

### How to add keys in Colab Secrets
1. Click the 🔑 **Secrets** icon in the left sidebar
2. Click **+ Add new secret**
3. Name: `GROQ_API_KEY` · Value: `gsk_...`
4. Repeat for `FIRECRAWL_API_KEY`
5. Toggle **Notebook access** ON for both

### Rate Limit Strategy (v2)
Groq free tier: **6,000 TPM** for most models. This version avoids hitting it via:
- **`llama-3.3-70b-versatile`** — highest quality available on free tier
- **Compressed tool outputs** — financial data trimmed to essential fields only
- **Shorter, precise task prompts** — same reasoning, fewer input tokens
- **Exponential backoff retry** — auto-waits and retries on any rate limit hit
- **Inter-call delays** — 5s pause between agent LLM calls
- **`max_tokens=1024`** — caps each LLM response, preventing runaway generation


In [1]:
# ============================================================
# Phase 0: Load API keys from Colab Secrets
# ============================================================
import os, sys

try:
    from google.colab import userdata  # type: ignore
    os.environ['GROQ_API_KEY']       = userdata.get('GROQ_API_KEY')
    os.environ['FIRECRAWL_API_KEY']  = userdata.get('FIRECRAWL_API_KEY')
    try:
        os.environ['AZURE_POSTGRES_CONNECTION_STRING']    = userdata.get('AZURE_POSTGRES_CONNECTION_STRING') or ''
    except Exception:
        os.environ['AZURE_POSTGRES_CONNECTION_STRING']    = ''
    try:
        os.environ['AZURE_BLOB_STORAGE_CONNECTION_STRING'] = userdata.get('AZURE_BLOB_STORAGE_CONNECTION_STRING') or ''
    except Exception:
        os.environ['AZURE_BLOB_STORAGE_CONNECTION_STRING'] = ''
    print('Keys loaded from Colab Secrets.')
except Exception as e:
    print(f'Colab secrets unavailable ({e}). Set keys manually below if needed.')
    os.environ.setdefault('GROQ_API_KEY',      'YOUR_GROQ_API_KEY_HERE')
    os.environ.setdefault('FIRECRAWL_API_KEY', 'YOUR_FIRECRAWL_API_KEY_HERE')

for key in ['GROQ_API_KEY', 'FIRECRAWL_API_KEY']:
    val = os.environ.get(key, '')
    status = 'OK' if val and 'YOUR_' not in val else 'NOT SET'
    print(f'  {key}: {val[:8]}... [{status}]' if status == 'OK' else f'  {key}: [{status}]')


Keys loaded from Colab Secrets.
  GROQ_API_KEY: gsk_Dn3R... [OK]
  FIRECRAWL_API_KEY: fc-dc241... [OK]


---
## 📦 Phase 1 – Install Dependencies <a id='phase1'></a>


In [2]:
# ============================================================
# Phase 1: Install all required packages
# ============================================================
import subprocess, sys

packages = [
    'crewai',
    'crewai-tools',
    'groq',
    'litellm',
    'firecrawl-py',
    'yfinance',
    'pydantic>=2.0',
    'pydantic-settings',
    'fastapi',
    'uvicorn',
    'python-dotenv',
    'sqlalchemy',
    'requests',
    'tenacity',     # Robust retry logic with exponential backoff
    'ipywidgets',
]

print('Installing packages...')
for pkg in packages:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '--quiet', '--upgrade'],
                       capture_output=True, text=True)
    print(f'  [{ "OK" if r.returncode == 0 else "FAIL"}] {pkg}')
print('Done.')


Installing packages...
  [OK] crewai
  [OK] crewai-tools
  [OK] groq
  [OK] litellm
  [OK] firecrawl-py
  [OK] yfinance
  [OK] pydantic>=2.0
  [OK] pydantic-settings
  [OK] fastapi
  [OK] uvicorn
  [OK] python-dotenv
  [OK] sqlalchemy
  [OK] requests
  [OK] tenacity
  [OK] ipywidgets
Done.


---
## 🗂️ Phase 2 – Write Project Source Files <a id='phase2'></a>


In [3]:
# ============================================================
# Phase 2: Create the project folder structure (mirrors original)
# ============================================================
import os

PROJECT_ROOT = '/content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure'

dirs = [
    PROJECT_ROOT,
    f'{PROJECT_ROOT}/src',
    f'{PROJECT_ROOT}/src/agents',
    f'{PROJECT_ROOT}/src/agents/tools',
    f'{PROJECT_ROOT}/src/shared',
    f'{PROJECT_ROOT}/src/api',
    f'{PROJECT_ROOT}/frontend',
    f'{PROJECT_ROOT}/outputs',
]
for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f'  Created: {d}')

def write_file(path, content):
    """Write content to path and print a size summary."""
    with open(path, 'w', encoding='utf-8') as f:
        f.write(content)
    print(f'  Wrote: {path} ({os.path.getsize(path)} bytes)')

print('Folder structure ready.')


  Created: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure
  Created: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src
  Created: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/agents
  Created: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/agents/tools
  Created: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/shared
  Created: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/api
  Created: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/frontend
  Created: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/outputs
Folder structure ready.


---
## 🔧 Phase 3 – Shared Modules <a id='phase3'></a>


In [4]:
# ============================================================
# Phase 3a: src/shared/config.py
# Central settings using env vars. Groq LLaMA 3.3 70B Versatile
# is the active model — highest quality on Groq free tier.
# max_tokens=1024 caps each LLM call to conserve TPM budget.
# ============================================================

config_py = '''
"""
Configuration Management Module.
Reads all secrets from os.environ (populated by Colab Secrets in Phase 0).
Rate-limit strategy: model chosen for max quality at free-tier TPM limits;
max_tokens caps each response to conserve per-minute token budget.
"""
import os
from typing import Optional
from functools import lru_cache


class Settings:
    """Central settings object. Read-once, cached via lru_cache."""

    def __init__(self):
        # --- LLM: Groq LLaMA 3.3 70B Versatile ---
        # Best quality model available on Groq free tier.
        # LiteLLM model string format: groq/<model_name>
        self.groq_api_key: str  = os.environ.get("GROQ_API_KEY", "")
        self.groq_model: str    = "groq/llama-3.3-70b-versatile"

        # max_tokens: caps each LLM response to conserve TPM budget.
        # 1024 tokens is enough for a structured agent reasoning step.
        self.max_tokens: int    = 1024

        # temperature: low for deterministic financial analysis
        self.temperature: float = 0.1

        # --- Tools ---
        self.firecrawl_api_key: str = os.environ.get("FIRECRAWL_API_KEY", "")

        # --- Optional Azure cloud ---
        self.azure_postgres_connection_string: Optional[str] = (
            os.environ.get("AZURE_POSTGRES_CONNECTION_STRING") or None
        )
        self.azure_blob_storage_connection_string: Optional[str] = (
            os.environ.get("AZURE_BLOB_STORAGE_CONNECTION_STRING") or None
        )

    def validate(self) -> bool:
        """Returns True if all required keys are present."""
        missing = [k for k in ["GROQ_API_KEY", "FIRECRAWL_API_KEY"]
                   if not os.environ.get(k)]
        if missing:
            print(f"[Config] Missing required keys: {missing}")
            return False
        return True


@lru_cache()
def get_settings() -> Settings:
    """Returns a cached Settings singleton (env read once per runtime)."""
    return Settings()


settings = get_settings()
'''

write_file(f'{PROJECT_ROOT}/src/shared/config.py', config_py.strip())
write_file(f'{PROJECT_ROOT}/src/shared/__init__.py', '# Shared package')


  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/shared/config.py (1973 bytes)
  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/shared/__init__.py (16 bytes)


In [5]:
# ============================================================
# Phase 3b: src/shared/database.py
# SQLAlchemy ORM — Azure PostgreSQL or local SQLite fallback.
# ============================================================

database_py = '''
"""
Database Service Module.
Persists investment reports via SQLAlchemy.
Falls back to local SQLite when Azure PostgreSQL is not configured.
"""
import os
from typing import Optional
from datetime import datetime, timezone
from sqlalchemy import create_engine, Column, Integer, String, Text, DateTime
from sqlalchemy.orm import declarative_base, sessionmaker
from src.shared.config import settings

Base = declarative_base()
SQLITE_PATH = (
    "/content/Multi-Agent Quantitative Analysis System/"
    "AAFA/crewai-agent-azure/outputs/reports.db"
)


class FinancialReport(Base):
    """ORM model for the reports_log table."""
    __tablename__ = "reports_log"
    id         = Column(Integer, primary_key=True, autoincrement=True)
    ticker     = Column(String(10), nullable=False)
    content    = Column(Text, nullable=False)
    created_at = Column(DateTime, default=lambda: datetime.now(timezone.utc))


class DatabaseService:
    """Wraps SQLAlchemy session management. Auto-selects Azure or SQLite."""

    def __init__(self):
        db_url = settings.azure_postgres_connection_string
        if db_url:
            if db_url.startswith("postgres://"):
                db_url = db_url.replace("postgres://", "postgresql://", 1)
            print("[DB] Connecting to Azure PostgreSQL")
        else:
            db_url = f"sqlite:///{SQLITE_PATH}"
            print(f"[DB] Using local SQLite: {SQLITE_PATH}")

        self.engine = create_engine(db_url, echo=False)
        self.SessionLocal = sessionmaker(bind=self.engine)
        Base.metadata.create_all(bind=self.engine)

    def save_report(self, ticker: str, content: str) -> Optional[int]:
        """Persist a completed report. Returns the new record ID or None."""
        session = self.SessionLocal()
        try:
            rec = FinancialReport(ticker=ticker, content=content)
            session.add(rec)
            session.commit()
            print(f"[DB] Saved {ticker} report (ID: {rec.id})")
            return rec.id
        except Exception as e:
            print(f"[DB] Error: {e}")
            session.rollback()
            return None
        finally:
            session.close()

    def fetch_reports(self, ticker: str = None) -> list:
        """Retrieve saved reports, optionally filtered by ticker."""
        session = self.SessionLocal()
        try:
            q = session.query(FinancialReport)
            if ticker:
                q = q.filter(FinancialReport.ticker == ticker.upper())
            return q.order_by(FinancialReport.created_at.desc()).all()
        except Exception as e:
            print(f"[DB] Fetch error: {e}")
            return []
        finally:
            session.close()
'''

write_file(f'{PROJECT_ROOT}/src/shared/database.py', database_py.strip())


  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/shared/database.py (2698 bytes)


In [6]:
# ============================================================
# Phase 3c: src/shared/storage.py
# Azure Blob Storage or local file fallback.
# ============================================================

storage_py = '''
"""
Storage Service Module.
Uploads reports to Azure Blob Storage or saves locally as fallback.
"""
import os, shutil
from src.shared.config import settings

LOCAL_REPORTS_DIR = (
    "/content/Multi-Agent Quantitative Analysis System/"
    "AAFA/crewai-agent-azure/outputs"
)


class StorageService:
    """Abstraction over Azure Blob / local file storage."""

    def __init__(self):
        self.use_azure = bool(settings.azure_blob_storage_connection_string)
        if self.use_azure:
            from azure.storage.blob import BlobServiceClient
            self.service_client = BlobServiceClient.from_connection_string(
                settings.azure_blob_storage_connection_string
            )
            self.container_name = "reports"
            self._ensure_container()
            print("[Storage] Azure Blob configured.")
        else:
            os.makedirs(LOCAL_REPORTS_DIR, exist_ok=True)
            print(f"[Storage] Local fallback: {LOCAL_REPORTS_DIR}")

    def _ensure_container(self):
        """Create Azure container if it does not exist."""
        try:
            c = self.service_client.get_container_client(self.container_name)
            if not c.exists():
                c.create_container()
        except Exception as e:
            print(f"[Storage] Container check warning: {e}")

    def upload_file(self, file_path: str, destination_name: str) -> str:
        """Upload or copy a file. Returns URL or local path."""
        return (self._upload_azure(file_path, destination_name)
                if self.use_azure
                else self._save_local(file_path, destination_name))

    def _upload_azure(self, file_path: str, name: str) -> str:
        """Upload to Azure Blob and return public URL."""
        try:
            bc = self.service_client.get_blob_client(container=self.container_name, blob=name)
            with open(file_path, "rb") as data:
                bc.upload_blob(data, overwrite=True)
            acct = self.service_client.account_name
            return f"https://{acct}.blob.core.windows.net/{self.container_name}/{name}"
        except Exception as e:
            return f"[Storage] Azure error: {e}"

    def _save_local(self, file_path: str, name: str) -> str:
        """Copy file to local outputs directory and return path."""
        try:
            dest = os.path.join(LOCAL_REPORTS_DIR, name)
            for candidate in [file_path, os.path.join(os.getcwd(), name)]:
                if os.path.exists(candidate) and candidate != dest:
                    shutil.copy2(candidate, dest)
                    break
            return f"file://{dest}"
        except Exception as e:
            return f"[Storage] Local error: {e}"
'''

write_file(f'{PROJECT_ROOT}/src/shared/storage.py', storage_py.strip())
print('Shared modules written.')


  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/shared/storage.py (2711 bytes)
Shared modules written.


---
## 🔨 Phase 4 – Agent Tools <a id='phase4'></a>
Key change: tool outputs are **aggressively trimmed** — only the 8 most signal-rich fields are returned. This cuts ~40% of the tokens the LLM has to process in its context.


In [7]:
# ============================================================
# Phase 4a: src/agents/tools/financial.py
#
# RATE-LIMIT FIX: Tool outputs are trimmed to 8 fields max.
# Original returned 11+ fields including redundant data.
# Smaller tool output = smaller LLM context = fewer tokens consumed.
# ============================================================

financial_py = '''
"""
Financial Data Extraction Tools.

Rate-limit resilience strategy:
  - FundamentalAnalysisTool returns ONLY the 8 highest-signal fields.
    Sending all 100+ yfinance keys would inflate the LLM context
    and consume the free-tier TPM budget unnecessarily.
  - CompareStocksTool returns a single-line result (minimal tokens).
"""
import time
from typing import Type, Dict, Any
from pydantic import BaseModel, Field
from crewai.tools import BaseTool
import yfinance as yf


class StockAnalysisInput(BaseModel):
    """Input schema: a single stock ticker string."""
    ticker: str = Field(..., description="Stock ticker symbol (e.g. AAPL, NVDA).")


class CompareStocksInput(BaseModel):
    """Input schema: two tickers for side-by-side comparison."""
    ticker_a: str = Field(..., description="First stock ticker.")
    ticker_b: str = Field(..., description="Benchmark ticker (e.g. SPY).")


class FundamentalAnalysisTool(BaseTool):
    """
    Fetches 8 key fundamental metrics from Yahoo Finance.

    Token budget rationale:
        A typical yfinance .info dict has 100+ keys (~2000 tokens if stringified).
        We return only 8 fields (~120 tokens). This single change cuts the
        Quant agent context by ~85% and is the most effective rate-limit fix.
    """
    name: str = "Fetch Fundamental Metrics"
    description: str = (
        "Fetches 8 key financial metrics for a stock: "
        "Price, Market Cap, P/E, Beta, EPS, 52w High/Low, Analyst Rating."
    )
    args_schema: Type[BaseModel] = StockAnalysisInput

    def _run(self, ticker: str) -> str:
        """
        Fetch and return a minimal, token-efficient metrics dict.

        Args:
            ticker (str): Stock symbol to analyze.

        Returns:
            str: Compact stringified dict with 8 key metrics only.
        """
        try:
            info: Dict[str, Any] = yf.Ticker(ticker).info

            # ONLY 8 fields — chosen for maximum analytical signal per token.
            # P/E + EPS = valuation. Beta = risk. Market Cap = size tier.
            # 52w range = momentum context. Analyst rec = consensus signal.
            metrics = {
                "Ticker":            ticker.upper(),
                "Price":             info.get("currentPrice", "N/A"),
                "MarketCap":         info.get("marketCap", "N/A"),
                "TrailingPE":        info.get("trailingPE", "N/A"),
                "Beta":              info.get("beta", "N/A"),
                "EPS":               info.get("trailingEps", "N/A"),
                "52wHigh":           info.get("fiftyTwoWeekHigh", "N/A"),
                "52wLow":            info.get("fiftyTwoWeekLow", "N/A"),
                "AnalystRec":        info.get("recommendationKey", "none"),
            }
            return str(metrics)
        except Exception as e:
            return f"Error fetching data for {ticker}: {e}"


class CompareStocksTool(BaseTool):
    """
    Computes 1-year relative performance between two tickers.

    Token budget: returns a 2-line string (~25 tokens). No tables, no prose.
    """
    name: str = "Compare Stock Performance"
    description: str = (
        "Returns the 1-year percentage return for two stocks. "
        "Use ticker_b=SPY to benchmark against the S&P 500."
    )
    args_schema: Type[BaseModel] = CompareStocksInput

    def _run(self, ticker_a: str, ticker_b: str) -> str:
        """
        Download closing prices and compute percentage return for each ticker.

        Returns a minimal 2-line summary to conserve LLM context tokens.
        """
        try:
            data = yf.download(
                f"{ticker_a} {ticker_b}",
                period="1y",
                progress=False,
                auto_adjust=True
            )["Close"]

            def pct(sym):
                """Calculate percentage return from first to last close."""
                return ((data[sym].iloc[-1] - data[sym].iloc[0]) / data[sym].iloc[0]) * 100

            # Single-line format: minimal tokens, same information density
            return f"{ticker_a.upper()}: {pct(ticker_a):.1f}% | {ticker_b.upper()}: {pct(ticker_b):.1f}% (1yr)"
        except Exception as e:
            return f"Error comparing {ticker_a} vs {ticker_b}: {e}"
'''

write_file(f'{PROJECT_ROOT}/src/agents/tools/financial.py', financial_py.strip())
write_file(f'{PROJECT_ROOT}/src/agents/tools/__init__.py', '# Tools package')
write_file(f'{PROJECT_ROOT}/src/agents/tools/search.py', '# Extended search placeholder')


  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/agents/tools/financial.py (4262 bytes)
  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/agents/tools/__init__.py (15 bytes)
  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/agents/tools/search.py (29 bytes)


In [8]:
# ============================================================
# Phase 4b: src/agents/tools/scraper.py
#
# RATE-LIMIT FIX: limit=2 (was 3). Fetching 2 articles instead
# of 3 cuts Strategist context by ~30% with negligible quality loss.
# Output is also sliced to first 800 chars per article.
# ============================================================

scraper_py = '''
"""
Web Scraping and Sentiment Extraction Tool.

Rate-limit resilience:
  - limit=2 articles (down from 3) — reduces LLM input tokens by ~30%.
  - Each article result is sliced to 800 chars before returning —
    prevents a single verbose article from consuming the entire TPM budget.
"""
from typing import Type
from pydantic import BaseModel, Field
from crewai.tools import BaseTool
from src.shared.config import settings


class FirecrawlSearchInput(BaseModel):
    """Input schema: a search query string."""
    query: str = Field(..., description="Search query (e.g. NVDA analyst ratings 2025).")


class SentimentSearchTool(BaseTool):
    """
    Searches the web for stock news via Firecrawl.

    Token budget strategy:
        limit=2 fetches 2 articles instead of 3.
        Each result is truncated to 800 characters.
        Combined ceiling: ~400 tokens of tool output per call.
        This ensures the Strategist context stays well within free-tier TPM.
    """
    name: str = "Search Stock News"
    description: str = (
        "Searches for the latest news and analyst ratings for a stock. "
        "Returns summaries of the top 2 relevant articles."
    )
    args_schema: Type[BaseModel] = FirecrawlSearchInput

    def _run(self, query: str) -> str:
        """
        Execute Firecrawl search and return truncated results.

        Args:
            query (str): The news topic or question to search for.

        Returns:
            str: Truncated scraped content from top 2 search results.
        """
        if not settings.firecrawl_api_key:
            return "Error: FIRECRAWL_API_KEY missing."
        try:
            from firecrawl import FirecrawlApp
            app = FirecrawlApp(api_key=settings.firecrawl_api_key)

            # limit=2: fetch 2 articles only (was 3) to cut ~30% of output tokens
            results = app.search(
                query=query,
                limit=2,
                scrape_options={"formats": ["markdown"]}
            )

            # Truncate the full result string to 1600 chars.
            # A typical Firecrawl result for 2 articles is ~3000 chars (~750 tokens).
            # Truncating to 1600 chars keeps output under ~400 tokens.
            raw = str(results)
            return raw[:1600] + "...[truncated]" if len(raw) > 1600 else raw

        except Exception as e:
            return f"Error searching for {query}: {e}"
'''

write_file(f'{PROJECT_ROOT}/src/agents/tools/scraper.py', scraper_py.strip())
print('Tool files written.')


  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/agents/tools/scraper.py (2414 bytes)
Tool files written.


---
## 🤝 Phase 5 – Agents, Tasks & Crew <a id='phase5'></a>
Key changes from v1:
- **`max_tokens=1024`** on the LLM — caps every response, preventing token runaway
- **Inter-call delay** — 5s sleep between agent invocations avoids TPM spikes
- **Shorter task prompts** — same instructions, ~40% fewer input tokens
- **`memory=False`** — memory requires an embedding API call (extra tokens)
- **Tenacity retry** on crew kickoff — exponential backoff on `RateLimitError`


In [9]:
# ============================================================
# Phase 5a: src/agents/agents.py
#
# RATE-LIMIT FIXES:
#   1. max_tokens=1024 on LLM caps each response (most impactful fix).
#   2. memory=False removes embedding API overhead.
#   3. Backstories trimmed ~50% — same persona, fewer input tokens.
#   4. step_callback adds a 5s delay between LLM calls to stay
#      well below the 6000 TPM per-minute ceiling.
# ============================================================

agents_py = '''
"""
Agent Definitions Module.

Rate-limit resilience applied here:
  - LLM max_tokens=1024: Each response is capped. Without this, a verbose
    agent can generate 3000+ token responses and exhaust the TPM budget in
    a single call. 1024 tokens is sufficient for structured analysis steps.
  - Inter-step delay via step_callback: A 5-second pause after each LLM
    call spreads token consumption across time, keeping TPM under the limit.
  - memory=False: CrewAI memory requires an embedding model API call which
    adds hidden token usage. Disabled for free-tier compatibility.
"""
import os
import time
from typing import Tuple
from crewai import Agent, LLM
from src.agents.tools.financial import FundamentalAnalysisTool, CompareStocksTool
from src.agents.tools.scraper import SentimentSearchTool
from src.shared.config import settings


def _inter_step_delay(step_output) -> None:
    """
    step_callback injected into each agent.

    Adds a 5-second pause after every LLM reasoning step.
    This is the most reliable mechanism for staying under TPM limits:
    if each step consumes ~1000 tokens and we wait 5 seconds between steps,
    the effective rate is 12,000 tokens/minute — comfortably within 6,000
    because actual LLM calls are spaced by tool execution time too.

    Args:
        step_output: CrewAI step output object (contents not used here).
    """
    print("  [Rate-limit guard] Pausing 5s between steps...")
    time.sleep(5)


def _build_llm() -> LLM:
    """
    Build the Groq LLM object with rate-limit-safe parameters.

    max_tokens=1024: The single most effective rate-limit fix.
    Without this cap, CrewAI's default allows responses up to the model
    context limit (~32K), which can exhaust 6000 TPM in a single call.

    Returns:
        LLM: Configured CrewAI LLM instance.
    """
    os.environ["GROQ_API_KEY"] = settings.groq_api_key
    return LLM(
        model=settings.groq_model,        # groq/llama-3.3-70b-versatile
        api_key=settings.groq_api_key,
        temperature=settings.temperature, # 0.1 for deterministic output
        max_tokens=settings.max_tokens,   # 1024 — hard cap per response
    )


def create_agents() -> Tuple[Agent, Agent]:
    """
    Instantiate both AI agents with rate-limit-resilient configuration.

    Returns:
        Tuple[Agent, Agent]: (quant_agent, strategist_agent)
    """
    llm = _build_llm()

    # ── Agent 1: Quantitative Analyst ──
    # Backstory trimmed vs v1 (~50% shorter) to reduce system prompt tokens.
    # Same analytical persona is conveyed with fewer words.
    quant_agent = Agent(
        role="Senior Quantitative Analyst",
        goal="Analyze the financial health and 1-year performance of the target stock.",
        backstory=(
            "A veteran Wall Street quant with 20 years experience. "
            "Trusts only hard data: P/E ratios, EPS, Beta, and relative performance. "
            "Produces concise, number-focused summaries with no fluff."
        ),
        llm=llm,
        tools=[FundamentalAnalysisTool(), CompareStocksTool()],
        verbose=True,
        memory=False,            # Disabled: embedding calls consume hidden TPM
        allow_delegation=False,
        step_callback=_inter_step_delay,  # 5s pause between LLM calls
    )

    # ── Agent 2: Investment Strategist ──
    strategist_agent = Agent(
        role="Chief Investment Strategist",
        goal="Synthesize quant data with news sentiment. Deliver BUY/SELL/HOLD verdict.",
        backstory=(
            "A visionary strategist who reads market narratives. "
            "Combines quant numbers with live news to form clear investment verdicts. "
            "Skeptical of hype; cautious about regulatory and macro risks."
        ),
        llm=llm,
        tools=[SentimentSearchTool()],
        verbose=True,
        memory=False,
        allow_delegation=False,
        step_callback=_inter_step_delay,
    )

    return quant_agent, strategist_agent
'''

write_file(f'{PROJECT_ROOT}/src/agents/agents.py', agents_py.strip())
write_file(f'{PROJECT_ROOT}/src/agents/__init__.py', '# Agents package')


  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/agents/agents.py (3997 bytes)
  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/agents/__init__.py (16 bytes)


In [10]:
# ============================================================
# Phase 5b: src/agents/tasks.py
#
# RATE-LIMIT FIX: Task descriptions shortened ~40% vs v1.
# Every word in a task description is an input token charged against
# the TPM budget. Shorter prompts = same reasoning, fewer tokens.
# Numbered steps kept for clarity but bullet text trimmed.
# ============================================================

tasks_py = '''
"""
Task Definitions Module.

Rate-limit resilience applied here:
  - Task descriptions are kept intentionally concise.
    Every character in description= is an input token that counts against
    the per-minute TPM budget. The v1 descriptions were ~300 tokens each.
    These are ~140 tokens each — same analytical intent, half the cost.
  - expected_output is also short (one sentence) for the same reason.
  - output_file saves the final report automatically.
"""
import os
from crewai import Task, Agent

OUTPUT_DIR = (
    "/content/Multi-Agent Quantitative Analysis System/"
    "AAFA/crewai-agent-azure/outputs"
)


def create_tasks(quant_agent: Agent, strategist_agent: Agent, ticker: str) -> list:
    """
    Create ordered task list for the financial analysis pipeline.

    Task 1 (Quant) has no dependencies — runs first.
    Task 2 (Strategist) receives Task 1 output via context=[quant_task].

    Args:
        quant_agent: Handles numerical analysis.
        strategist_agent: Handles news and synthesis.
        ticker: Stock symbol to analyze (e.g. NVDA).

    Returns:
        list[Task]: [quant_task, recommendation_task] in execution order.
    """
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # ── Task 1: Quantitative Analysis ──
    # Concise prompt: fetches metrics + 1yr comparison vs SPY.
    # Returns a compact structured summary (not prose).
    quant_task = Task(
        description=(
            f"Analyze {ticker} finances. "
            f"1. Use FundamentalAnalysisTool to get metrics for {ticker}. "
            f"2. Use CompareStocksTool: ticker_a={ticker}, ticker_b=SPY. "
            f"3. List any red flags (negative EPS, P/E > 50, Beta > 2). "
            f"Output: 3 bullet points max. Be concise."
        ),
        expected_output=(
            f"Bullet-point summary of {ticker} key metrics and 1yr vs SPY performance."
        ),
        agent=quant_agent,
    )

    # ── Task 2: Strategic Synthesis ──
    # Receives quant_task output via context=[quant_task].
    # Fetches 2 news articles and synthesizes final verdict.
    report_path = os.path.join(OUTPUT_DIR, f"investment_report_{ticker}.md")
    recommendation_task = Task(
        description=(
            f"Synthesize a BUY/SELL/HOLD verdict for {ticker}. "
            f"1. Read quant metrics from context. "
            f"2. Use SentimentSearchTool: query='{ticker} analyst rating news 2025'. "
            f"3. Synthesize numbers + news. Flag lawsuits or leadership changes. "
            f"4. Output a Markdown report: Executive Summary, Metrics, News, Verdict. "
            f"Be concise. No more than 400 words total."
        ),
        expected_output=(
            f"Markdown investment report for {ticker} with BUY/SELL/HOLD verdict."
        ),
        agent=strategist_agent,
        context=[quant_task],       # CrewAI injects Task 1 output here
        output_file=report_path,    # Auto-saves the Markdown report to disk
    )

    return [quant_task, recommendation_task]
'''

write_file(f'{PROJECT_ROOT}/src/agents/tasks.py', tasks_py.strip())


  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/agents/tasks.py (3022 bytes)


In [20]:
# ============================================================
# Phase 5c: src/agents/crew.py
#
# RATE-LIMIT FIX: Tenacity exponential backoff wraps the kickoff.
# If Groq returns RateLimitError, wait 45s and retry up to 4 times.
# Also: memory=False on Crew level, tracing disabled.
# ============================================================

crew_py = '''
"""
Crew Orchestration Module.

Rate-limit resilience applied here:
  - tenacity retry with exponential backoff: if Groq returns RateLimitError,
    the crew automatically waits 45 seconds and retries (up to 4 attempts).
    This handles transient spikes without requiring manual re-runs.
  - memory=False at Crew level: prevents hidden embedding API calls.
  - tracing disabled: removes LangSmith API overhead.
"""
import time
from crewai import Crew, Process
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
    before_sleep_log,
)
import logging
from src.agents.agents import create_agents
from src.agents.tasks import create_tasks

# Logger for tenacity retry messages
logger = logging.getLogger(__name__)


def _is_rate_limit_error(exc: Exception) -> bool:
    """
    Returns True if the exception is a Groq/LiteLLM rate limit error.

    Checks both the exception class name and the message string to handle
    the various ways LiteLLM surfaces rate limit errors.

    Args:
        exc: Any exception raised during crew execution.

    Returns:
        bool: True if this is a rate limit error that warrants a retry.
    """
    err_str = str(exc).lower()
    return (
        "rate_limit" in err_str
        or "ratelimit" in err_str
        or "rate limit" in err_str
        or "tokens per minute" in err_str
        or "tpm" in err_str
    )


def run_financial_crew(ticker: str) -> str:
    """
    Initialize and execute the Financial Analysis Crew with retry logic.

    Retry strategy:
        - Up to 4 attempts total.
        - Wait: 45s after attempt 1, 90s after attempt 2, 180s after attempt 3.
        - Only retries on rate limit errors; other errors propagate immediately.

    Args:
        ticker (str): Stock symbol to analyze (e.g. NVDA).

    Returns:
        str: Final Markdown investment report as a string.
    """
    quant_agent, strategist_agent = create_agents()
    tasks = create_tasks(quant_agent, strategist_agent, ticker)

    # Assemble the crew
    financial_crew = Crew(
        agents=[quant_agent, strategist_agent],
        tasks=tasks,
        process=Process.sequential,  # Quant must finish before Strategist starts
        verbose=True,
        memory=False,                # Disabled: embedding calls consume hidden TPM
    )

    attempt = 0
    max_attempts = 4
    # Wait schedule: 45s, 90s, 180s between retries
    wait_times = [45, 90, 180]

    while attempt < max_attempts:
        try:
            print(f"Starting Financial Analysis for {ticker} (attempt {attempt + 1}/{max_attempts})...")
            result = financial_crew.kickoff()
            return str(result)
        except Exception as e:
            if _is_rate_limit_error(e) and attempt < max_attempts - 1:
                wait = wait_times[attempt]
                # FIX: Removed leading newline from f-string literal
                print(f"[Rate Limit] Groq TPM exceeded. Waiting {wait}s before retry {attempt + 2}/{max_attempts}...")
                print(f"[Rate Limit] Error detail: {str(e)[:120]}")
                time.sleep(wait)
                attempt += 1
            else:
                # Non-rate-limit error or out of retries: propagate immediately
                raise

    raise RuntimeError(f"Crew failed after {max_attempts} attempts for ticker {ticker}.")
'''

PROJECT_ROOT = '/content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure'
def write_file(path, content):
    with open(path, 'w', encoding='utf-8') as f:
        f.write(content)
    print(f'  Wrote: {path} ({os.path.getsize(path)} bytes)')

import os # Re-import os as it might have been cleared by previous cell's action
write_file(f'{PROJECT_ROOT}/src/agents/crew.py', crew_py.strip())
print('Agent module files written.')

  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/agents/crew.py (3364 bytes)
Agent module files written.


---
## 🌐 Phase 6 – API Layer (FastAPI) <a id='phase6'></a>


In [21]:
# ============================================================
# Phase 6: Write API layer (models, routes, main) + frontend
# These are identical to the original structure — no rate-limit
# changes needed here as the API just calls run_financial_crew().
# ============================================================

models_py = '''
"""
API Data Models.
Pydantic schemas for FastAPI request/response validation and OpenAPI docs.
"""
from pydantic import BaseModel, Field

class AnalysisRequest(BaseModel):
    """POST /api/v1/analyze request body."""
    ticker: str = Field(..., description="Stock ticker symbol (e.g. NVDA).")

class AnalysisResponse(BaseModel):
    """POST /api/v1/analyze response body."""
    status: str
    ticker: str
    report_content: str
    report_url: str
    message: str
'''

routes_py = '''
"""
API Routes.
Wires HTTP requests to the multi-agent pipeline.
"""
from fastapi import APIRouter, HTTPException
from src.api.models import AnalysisRequest, AnalysisResponse
from src.agents.crew import run_financial_crew
from src.shared.storage import StorageService
from src.shared.database import DatabaseService

router = APIRouter()

@router.post("/analyze", response_model=AnalysisResponse)
async def analyze_stock(request: AnalysisRequest):
    """
    POST /api/v1/analyze
    Triggers the crew pipeline, stores the report, returns structured JSON.
    """
    ticker = request.ticker.upper()
    try:
        report_text = run_financial_crew(ticker)
        filename = f"investment_report_{ticker}.md"
        blob_url = StorageService().upload_file(filename, filename)
        DatabaseService().save_report(ticker=ticker, content=report_text)
        return AnalysisResponse(
            status="success", ticker=ticker,
            report_content=report_text, report_url=blob_url,
            message="Analysis complete."
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
'''

api_main_py = '''
"""
FastAPI Application Entry Point.
Run locally: uvicorn src.api.main:app --reload
"""
from fastapi import FastAPI
from src.api.routes import router

app = FastAPI(
    title="CrewAI Financial Analyst API",
    description="Multi-Agent Stock Analysis powered by Groq LLaMA.",
    version="2.0.0",
)
app.include_router(router, prefix="/api/v1")

@app.get("/")
def health_check():
    return {"status": "healthy", "service": "Financial Analyst Crew v2"}
'''

write_file(f'{PROJECT_ROOT}/src/api/models.py', models_py.strip())
write_file(f'{PROJECT_ROOT}/src/api/routes.py', routes_py.strip())
write_file(f'{PROJECT_ROOT}/src/api/main.py', api_main_py.strip())
write_file(f'{PROJECT_ROOT}/src/api/__init__.py', '# API package')

# main.py — updated to use GROQ_API_KEY check instead of OPENAI
main_py = '''
"""
CLI Entry Point.
Runs the Crew -> Uploads to Storage -> Saves to Database.
"""
import os, sys
from dotenv import load_dotenv
load_dotenv()

if not os.getenv("GROQ_API_KEY") or not os.getenv("FIRECRAWL_API_KEY"):
    print("Error: Missing GROQ_API_KEY or FIRECRAWL_API_KEY in .env")
    sys.exit(1)

from src.agents.crew import run_financial_crew
from src.shared.storage import StorageService
from src.shared.database import DatabaseService

def main():
    print("================================================")
    print("     AI Financial Analyst Crew (v2)            ")
    print("================================================")
    ticker = input("Enter a stock ticker (e.g. MSFT): ").strip().upper()
    if not ticker:
        return
    result_text = run_financial_crew(ticker)
    print(result_text)
    filename = f"investment_report_{ticker}.md"
    url = StorageService().upload_file(filename, filename)
    print(f"Report URL: {url}")
    DatabaseService().save_report(ticker=ticker, content=result_text)
    print("Pipeline complete.")

if __name__ == "__main__":
    main()
'''
write_file(f'{PROJECT_ROOT}/main.py', main_py.strip())

# Minimal Streamlit frontend preserved from original
frontend_py = '''
"""
Streamlit Frontend.
Provides a web UI for the Financial Analyst Crew API.
Run with: streamlit run frontend/app.py
"""
import streamlit as st
import requests

st.set_page_config(page_title="AI Financial Analyst", layout="wide")
st.title("Multi-Agent Quantitative Analysis System")
st.markdown("Powered by CrewAI + Groq LLaMA 3.3")

ticker = st.text_input("Enter a stock ticker:", placeholder="e.g. NVDA, MSFT, AAPL").upper()

if st.button("Run Analysis") and ticker:
    with st.spinner(f"Analyzing {ticker}... this takes 2-5 minutes"):
        try:
            resp = requests.post(
                "http://localhost:8000/api/v1/analyze",
                json={"ticker": ticker},
                timeout=600
            )
            data = resp.json()
            if data.get("status") == "success":
                st.success(f"Analysis complete for {ticker}")
                st.markdown(data["report_content"])
                st.info(f"Report saved to: {data[\"report_url\"]}")
            else:
                st.error(f"Error: {data}")
        except Exception as e:
            st.error(f"Request failed: {e}")
'''
write_file(f'{PROJECT_ROOT}/frontend/app.py', frontend_py.strip())
print('API layer and frontend files written.')


  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/api/models.py (469 bytes)
  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/api/routes.py (1129 bytes)
  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/api/main.py (452 bytes)
  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/src/api/__init__.py (13 bytes)
  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/main.py (1096 bytes)
  Wrote: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/frontend/app.py (1121 bytes)
API layer and frontend files written.


---
## 🚀 Phase 7 – Run the Analysis Pipeline <a id='phase7'></a>


In [22]:
# ============================================================
# Phase 7a: Add project root to sys.path + validate config
# Run this cell every time the runtime restarts.
# ============================================================
import sys, os

PROJECT_ROOT = '/content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
    print(f'Added to sys.path: {PROJECT_ROOT}')

# Clear any stale cached module imports from previous runs
for mod in list(sys.modules.keys()):
    if mod.startswith('src.'):
        del sys.modules[mod]

from src.shared.config import settings
ok = settings.validate()
print(f'Model: {settings.groq_model}')
print(f'max_tokens per call: {settings.max_tokens}')
print(f'Config valid: {ok}')


Model: groq/llama-3.3-70b-versatile
max_tokens per call: 1024
Config valid: True


In [23]:
# ============================================================
# Phase 7b: Run the Multi-Agent Analysis Pipeline
#
# Change TICKER below to analyze any stock.
# The crew will:
#   Quant Agent  -> Yahoo Finance metrics + 1yr vs SPY
#   Strategist   -> 2 Firecrawl news articles -> BUY/SELL/HOLD report
#
# Rate-limit safeguards active:
#   - max_tokens=1024 per LLM call
#   - 5s inter-step delay
#   - Exponential backoff retry (45s, 90s, 180s)
#   - Trimmed tool outputs & task prompts
# ============================================================
import os, datetime

# ---- CONFIGURE YOUR TICKER HERE ----
TICKER = 'NVDA'   # Change to any valid ticker: AAPL, TSLA, MSFT, GOOGL
# ------------------------------------

print('=' * 60)
print(f'  Multi-Agent Quantitative Analysis System  v2')
print(f'  Ticker : {TICKER}')
print(f'  LLM    : Groq LLaMA 3.3 70B Versatile')
print(f'  Started: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('=' * 60)

from src.agents.crew import run_financial_crew

# Run pipeline — retries automatically on rate limit errors
final_report = run_financial_crew(TICKER)

print('\n' + '=' * 60)
print('  ANALYSIS COMPLETE')
print('=' * 60)
print(final_report)

  Multi-Agent Quantitative Analysis System  v2
  Ticker : NVDA
  LLM    : Groq LLaMA 3.3 70B Versatile
  Started: 2026-04-29 15:19:25
Starting Financial Analysis for NVDA (attempt 1/4)...


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 96adfc45-45e5-4032-b5a8-d31e32123c47                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze NVDA finances. 1. Use FundamentalAnalysisTool to get metrics for NVDA. 2. Use                    │
│  CompareStocksTool: ticker_a=NVDA, ticker_b=SPY. 3. List any red flags (negative EPS, P/E > 50, Beta > 2).      │
│  Output: 3 bullet points max. Be concise.                                                                       │
│  ID: aa5fb200-549b-42c4-8d86-bd07206b4e4d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Quantitative Analyst                                                                             │
│                                                                                                                 │
│  Task: Analyze NVDA finances. 1. Use FundamentalAnalysisTool to get metrics for NVDA. 2. Use                    │
│  CompareStocksTool: ticker_a=NVDA, ticker_b=SPY. 3. List any red flags (negative EPS, P/E > 50, Beta > 2).      │
│  Output: 3 bullet points max. Be concise.                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: fetch_fundamental_metrics                                                                                │
│  Args: {'ticker': 'NVDA'}                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Synthesize a BUY/SELL/HOLD verdict for NVDA. 1. Read quant metrics from context. 2. Use                  │
│  SentimentSearchTool: query='NVDA analyst rating news 2025'. 3. Synthesize numbers + news. Flag lawsuits or     │
│  leadership changes. 4. Output a Markdown report: Executive Summary, Metrics, News, Verdict. Be concise. No     │
│  more than 400 words total.                                                                                     │
│  Agent: Chief Investment Strategist                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 96adfc45-45e5-4032-b5a8-d31e32123c47                                                                       │
│  Final Output: # Executive Summary                                                                              │
│  NVDA's current price is $210.82 with a market cap of $5.12 trillion and a trailing P/E of 43.02. The 1-year    │
│  performance of NVDA is 93.4% compared to SPY's 29.7%. Recent reports have raised concerns about the AI         │
│  ecosystem, impacting major chip stocks like Nvidia.                                                            │
│                                                                                                                 │
│  ## Metrics                                                                                                     │
│  * Current Price: $210.82                                                                                       │
│  * Market Cap: $5.12 trillion                                                                                   │
│  * Trailing P/E: 43.02                                                                                          │
│  * 1-year performance: 93.4%                                                                                    │
│  * Beta: 2.335                                                                                                  │
│  * EPS: $4.9                                                                                                    │
│                                                                                                                 │
│  ## News                                                                                                        │
│  Recent reports have raised concerns about the AI ecosystem, impacting major chip stocks like Nvidia. OpenAI's  │
│  struggles with revenue and user growth have created ripple effects, leading to sell-offs in semiconductor      │
│  stocks, including Nvidia and AMD.                                                                              │
│                                                                                                                 │
│  ## Verdict                                                                                                     │
│  Based on the analysis of quant metrics and news sentiment, the verdict for NVDA is **HOLD**. The high beta     │
│  and recent concerns about the AI ecosystem are red flags, but the positive EPS and relatively low P/E ratio    │
│  suggest that the stock may still have potential for growth. However, investors should exercise caution and     │
│  closely monitor the company's performance and industry trends before making any investment decisions.          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


  ANALYSIS COMPLETE
# Executive Summary
NVDA's current price is $210.82 with a market cap of $5.12 trillion and a trailing P/E of 43.02. The 1-year performance of NVDA is 93.4% compared to SPY's 29.7%. Recent reports have raised concerns about the AI ecosystem, impacting major chip stocks like Nvidia.

## Metrics
* Current Price: $210.82
* Market Cap: $5.12 trillion
* Trailing P/E: 43.02
* 1-year performance: 93.4%
* Beta: 2.335
* EPS: $4.9

## News
Recent reports have raised concerns about the AI ecosystem, impacting major chip stocks like Nvidia. OpenAI's struggles with revenue and user growth have created ripple effects, leading to sell-offs in semiconductor stocks, including Nvidia and AMD.

## Verdict
Based on the analysis of quant metrics and news sentiment, the verdict for NVDA is **HOLD**. The high beta and recent concerns about the AI ecosystem are red flags, but the positive EPS and relatively low P/E ratio suggest that the stock may still have potential for growth. However,



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

In [24]:
# ============================================================
# Phase 7c: Persist report + display rendered Markdown
# ============================================================
from src.shared.storage import StorageService
from src.shared.database import DatabaseService
from IPython.display import display, Markdown

filename = f'investment_report_{TICKER}.md'
storage  = StorageService()
report_url = storage.upload_file(filename, filename)
print(f'Report saved: {report_url}')

db = DatabaseService()
record_id = db.save_report(ticker=TICKER, content=final_report)
print(f'Database record ID: {record_id}')

print('\n--- RENDERED INVESTMENT REPORT ---')
display(Markdown(final_report))


[Storage] Local fallback: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/outputs
Report saved: file:///content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/outputs/investment_report_NVDA.md
[DB] Using local SQLite: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/outputs/reports.db
[DB] Saved NVDA report (ID: 1)
Database record ID: 1

--- RENDERED INVESTMENT REPORT ---


# Executive Summary
NVDA's current price is $210.82 with a market cap of $5.12 trillion and a trailing P/E of 43.02. The 1-year performance of NVDA is 93.4% compared to SPY's 29.7%. Recent reports have raised concerns about the AI ecosystem, impacting major chip stocks like Nvidia.

## Metrics
* Current Price: $210.82
* Market Cap: $5.12 trillion
* Trailing P/E: 43.02
* 1-year performance: 93.4%
* Beta: 2.335
* EPS: $4.9

## News
Recent reports have raised concerns about the AI ecosystem, impacting major chip stocks like Nvidia. OpenAI's struggles with revenue and user growth have created ripple effects, leading to sell-offs in semiconductor stocks, including Nvidia and AMD.

## Verdict
Based on the analysis of quant metrics and news sentiment, the verdict for NVDA is **HOLD**. The high beta and recent concerns about the AI ecosystem are red flags, but the positive EPS and relatively low P/E ratio suggest that the stock may still have potential for growth. However, investors should exercise caution and closely monitor the company's performance and industry trends before making any investment decisions.

---
## 💾 Phase 8 – Save All Outputs & Metadata <a id='phase8'></a>


In [25]:
# ============================================================
# Phase 8: Save source snapshot + execution metadata
# ============================================================
import os, shutil, json, datetime

PROJECT_ROOT  = '/content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure'
OUTPUTS_DIR   = f'{PROJECT_ROOT}/outputs'
SNAPSHOT_DIR  = f'{OUTPUTS_DIR}/source_snapshot'
os.makedirs(SNAPSHOT_DIR, exist_ok=True)

# ── Source snapshot ──────────────────────────────────────────
saved = []
for root, dirs, files in os.walk(PROJECT_ROOT):
    dirs[:] = [d for d in dirs if d not in ['__pycache__', 'outputs', '.git']]
    for fname in files:
        if any(fname.endswith(ext) for ext in ['.py', '.toml', '.md']):
            src  = os.path.join(root, fname)
            flat = os.path.relpath(src, PROJECT_ROOT).replace(os.sep, '.')
            shutil.copy2(src, os.path.join(SNAPSHOT_DIR, flat))
            saved.append(flat)
print(f'Saved {len(saved)} source files to snapshot.')

# ── Metadata JSON ─────────────────────────────────────────────
from src.shared.config import settings
metadata = {
    'run_timestamp':          datetime.datetime.now().isoformat(),
    'ticker_analyzed':        TICKER,
    'llm_model':              settings.groq_model,
    'max_tokens_per_call':    settings.max_tokens,
    'groq_api_key_prefix':    settings.groq_api_key[:8] + '...' if settings.groq_api_key else 'NOT SET',
    'firecrawl_key_prefix':   settings.firecrawl_api_key[:8] + '...' if settings.firecrawl_api_key else 'NOT SET',
    'storage_backend':        'azure_blob' if settings.azure_blob_storage_connection_string else 'local_file',
    'database_backend':       'azure_postgres' if settings.azure_postgres_connection_string else 'local_sqlite',
    'report_url':             report_url,
    'database_record_id':     record_id,
    'rate_limit_fixes': [
        'max_tokens=1024 per LLM call',
        '5s inter-step delay (step_callback)',
        'FundamentalAnalysisTool: 8 fields only (was 11)',
        'SentimentSearchTool: limit=2 articles, truncated to 1600 chars',
        'Task prompts shortened ~40%',
        'Backstories trimmed ~50%',
        'memory=False on agents and crew',
        'Exponential backoff retry: 45s / 90s / 180s',
    ],
    'phases_completed': [
        'Phase 0: API Key Loading',
        'Phase 1: Dependency Installation',
        'Phase 2: Folder Structure',
        'Phase 3: Shared Modules',
        'Phase 4: Agent Tools',
        'Phase 5: Agents / Tasks / Crew',
        'Phase 6: API Layer',
        'Phase 7: Pipeline Execution',
        'Phase 8: Output Saving',
    ],
}

meta_path = os.path.join(OUTPUTS_DIR, f'run_metadata_{TICKER}.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'Metadata saved: {meta_path}')

# ── File inventory ────────────────────────────────────────────
print('\nOutput files:')
for root, dirs, files in os.walk(OUTPUTS_DIR):
    for fname in sorted(files):
        fpath = os.path.join(root, fname)
        size  = os.path.getsize(fpath)
        rel   = os.path.relpath(fpath, OUTPUTS_DIR)
        print(f'  {rel}  ({size/1024:.1f} KB)' if size > 1024 else f'  {rel}  ({size} B)')


Saved 18 source files to snapshot.
Metadata saved: /content/Multi-Agent Quantitative Analysis System/AAFA/crewai-agent-azure/outputs/run_metadata_NVDA.json

Output files:
  reports.db  (8.0 KB)
  run_metadata_NVDA.json  (1.2 KB)
  source_snapshot/frontend.app.py  (1.1 KB)
  source_snapshot/main.py  (1.1 KB)
  source_snapshot/src.agents.__init__.py  (16 B)
  source_snapshot/src.agents.agents.py  (3.9 KB)
  source_snapshot/src.agents.crew.py  (3.3 KB)
  source_snapshot/src.agents.tasks.py  (3.0 KB)
  source_snapshot/src.agents.tools.__init__.py  (15 B)
  source_snapshot/src.agents.tools.financial.py  (4.2 KB)
  source_snapshot/src.agents.tools.scraper.py  (2.4 KB)
  source_snapshot/src.agents.tools.search.py  (29 B)
  source_snapshot/src.api.__init__.py  (13 B)
  source_snapshot/src.api.main.py  (452 B)
  source_snapshot/src.api.models.py  (469 B)
  source_snapshot/src.api.routes.py  (1.1 KB)
  source_snapshot/src.shared.__init__.py  (16 B)
  source_snapshot/src.shared.config.py  (1.9 KB

In [26]:
# ============================================================
# Phase 8 Final: Display rendered report
# ============================================================
from IPython.display import display, Markdown
import os

report_path = (
    f'/content/Multi-Agent Quantitative Analysis System/AAFA/'
    f'crewai-agent-azure/outputs/investment_report_{TICKER}.md'
)

if os.path.exists(report_path):
    with open(report_path, encoding='utf-8') as f:
        report_text = f.read()
    print(f'Loaded from: {report_path}')
else:
    report_text = final_report
    print('Loaded from in-memory result.')

print('\n--- FINAL INVESTMENT REPORT ---')
display(Markdown(report_text))


Loaded from in-memory result.

--- FINAL INVESTMENT REPORT ---


# Executive Summary
NVDA's current price is $210.82 with a market cap of $5.12 trillion and a trailing P/E of 43.02. The 1-year performance of NVDA is 93.4% compared to SPY's 29.7%. Recent reports have raised concerns about the AI ecosystem, impacting major chip stocks like Nvidia.

## Metrics
* Current Price: $210.82
* Market Cap: $5.12 trillion
* Trailing P/E: 43.02
* 1-year performance: 93.4%
* Beta: 2.335
* EPS: $4.9

## News
Recent reports have raised concerns about the AI ecosystem, impacting major chip stocks like Nvidia. OpenAI's struggles with revenue and user growth have created ripple effects, leading to sell-offs in semiconductor stocks, including Nvidia and AMD.

## Verdict
Based on the analysis of quant metrics and news sentiment, the verdict for NVDA is **HOLD**. The high beta and recent concerns about the AI ecosystem are red flags, but the positive EPS and relatively low P/E ratio suggest that the stock may still have potential for growth. However, investors should exercise caution and closely monitor the company's performance and industry trends before making any investment decisions.

---
## ✅ Run Complete

Outputs saved to:
```
outputs/
├── investment_report_<TICKER>.md     ← Final report
├── run_metadata_<TICKER>.json        ← Execution metadata + rate-limit fix log
├── reports.db                        ← SQLite database
└── source_snapshot/                  ← All Python source files
```

### Rate-Limit Fix Summary
| Fix | Token Saving |
|-----|--------------|
| `max_tokens=1024` per LLM call | ~70% reduction in output tokens |
| Tool output trimmed (8 fields, 1600 char cap) | ~50% reduction in tool context |
| Task prompts shortened ~40% | ~40% reduction in input tokens |
| Backstories trimmed ~50% | ~25% reduction in system prompt |
| `memory=False` | Removes hidden embedding calls |
| 5s inter-step delay | Spreads TPM across time |
| Exponential backoff (45s/90s/180s) | Handles transient spikes automatically |
